In [1]:
import numpy as np
import pandas as pd

Importing Data set

In [2]:
df = pd.read_excel('../data/ba_schedule.xlsx')

In [3]:
df.head()

,FLIGHT_DATE,FLIGHT_TIME,TIME_OF_DAY,AIRLINE_CD,FLIGHT_NO,DEPARTURE_STATION_CD,ARRIVAL_STATION_CD,ARRIVAL_COUNTRY,ARRIVAL_REGION,HAUL,AIRCRAFT_TYPE,FIRST_CLASS_SEATS,BUSINESS_CLASS_SEATS,ECONOMY_SEATS,TIER1_ELIGIBLE_PAX,TIER2_ELIGIBLE_PAX,TIER3_ELIGIBLE_PAX
0,2025-09-02,14:19:00,Afternoon,BA,BA5211,LHR,LAX,USA,North America,LONG,B777,8,49,178,0,10,38
1,2025-06-10,06:42:00,Morning,BA,BA7282,LHR,LAX,USA,North America,LONG,B777,8,49,178,0,7,28
2,2025-10-27,15:33:00,Afternoon,BA,BA1896,LHR,FRA,Germany,Europe,SHORT,A320,0,17,163,0,11,40
3,2025-06-15,18:29:00,Evening,BA,BA5497,LHR,IST,Turkey,Europe,SHORT,A320,0,8,172,0,16,54
4,2025-08-25,20:35:00,Evening,BA,BA1493,LHR,FRA,Germany,Europe,SHORT,A320,0,13,167,0,6,27


In [4]:
cat_cols = df.select_dtypes(include='str').columns
df.loc[:,cat_cols] = df[cat_cols].astype('category')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   FLIGHT_DATE           10000 non-null  datetime64[us]
 1   FLIGHT_TIME           10000 non-null  object        
 2   TIME_OF_DAY           10000 non-null  str           
 3   AIRLINE_CD            10000 non-null  str           
 4   FLIGHT_NO             10000 non-null  str           
 5   DEPARTURE_STATION_CD  10000 non-null  str           
 6   ARRIVAL_STATION_CD    10000 non-null  str           
 7   ARRIVAL_COUNTRY       10000 non-null  str           
 8   ARRIVAL_REGION        10000 non-null  str           
 9   HAUL                  10000 non-null  str           
 10  AIRCRAFT_TYPE         10000 non-null  str           
 11  FIRST_CLASS_SEATS     10000 non-null  int64         
 12  BUSINESS_CLASS_SEATS  10000 non-null  int64         
 13  ECONOMY_SEATS         10000 

In [5]:
ba_df = df.loc[:,cat_cols]
print(ba_df.describe())


       TIME_OF_DAY AIRLINE_CD FLIGHT_NO DEPARTURE_STATION_CD  \
count        10000      10000     10000                10000   
unique           4          1      6037                    1   
top        Morning         BA    BA8347                  LHR   
freq          3530      10000         7                10000   

       ARRIVAL_STATION_CD ARRIVAL_COUNTRY ARRIVAL_REGION   HAUL AIRCRAFT_TYPE  
count               10000           10000          10000  10000         10000  
unique                 15              10              4      2             5  
top                   FRA             USA         Europe  SHORT          A320  
freq                  714            2658           5975   5975          5975  


In [6]:
print(df.query("TIER1_ELIGIBLE_PAX > 0")['TIER1_ELIGIBLE_PAX'].count())

3768


In [7]:
df['TOTAL_PASSENGERS'] = df[['FIRST_CLASS_SEATS','BUSINESS_CLASS_SEATS','ECONOMY_SEATS']].sum(axis=1)
df.head()

,FLIGHT_DATE,FLIGHT_TIME,TIME_OF_DAY,AIRLINE_CD,FLIGHT_NO,DEPARTURE_STATION_CD,ARRIVAL_STATION_CD,ARRIVAL_COUNTRY,ARRIVAL_REGION,HAUL,AIRCRAFT_TYPE,FIRST_CLASS_SEATS,BUSINESS_CLASS_SEATS,ECONOMY_SEATS,TIER1_ELIGIBLE_PAX,TIER2_ELIGIBLE_PAX,TIER3_ELIGIBLE_PAX,TOTAL_PASSENGERS
0,2025-09-02,14:19:00,Afternoon,BA,BA5211,LHR,LAX,USA,North America,LONG,B777,8,49,178,0,10,38,235
1,2025-06-10,06:42:00,Morning,BA,BA7282,LHR,LAX,USA,North America,LONG,B777,8,49,178,0,7,28,235
2,2025-10-27,15:33:00,Afternoon,BA,BA1896,LHR,FRA,Germany,Europe,SHORT,A320,0,17,163,0,11,40,180
3,2025-06-15,18:29:00,Evening,BA,BA5497,LHR,IST,Turkey,Europe,SHORT,A320,0,8,172,0,16,54,180
4,2025-08-25,20:35:00,Evening,BA,BA1493,LHR,FRA,Germany,Europe,SHORT,A320,0,13,167,0,6,27,180


In [20]:

def get_tier_proportion(df,group_by,tier_eligible_cols):
    
    #Transformation to group and sum the tiers
    proportion_group = df.groupby(group_by)[tier_eligible_cols].transform('sum')
    print(proportion_group)
    proportion_group['total_pax'] = proportion_group[tier_eligible_cols].sum(axis=1)
    proportion_group[tier_eligible_cols] = proportion_group[tier_eligible_cols].div(proportion_group['total_pax'],axis=0).mul(100)
    #Find porportions by dividing by the total number of passengers
    #proportion_group[tier_eligible_cols] = proportion_group[tier_eligible_cols].div(proportion_group[total_passenger_col],axis=0).mul(100)
    #Create new column names for our transformed columns
    new_col_names = []
    for i in range(len(tier_eligible_cols)):
        new_col_names.append(f'TIER_{i+1}_%')
    new_col_names.append('total_pax')
    #Rename columns
    proportion_group.columns = new_col_names
    return proportion_group.round(1)

In [21]:
tiers_cols = ['TIER1_ELIGIBLE_PAX','TIER2_ELIGIBLE_PAX','TIER3_ELIGIBLE_PAX']
proportion_time = get_tier_proportion(df,['HAUL','TIME_OF_DAY'],tiers_cols)
proportion_time

      TIER1_ELIGIBLE_PAX  TIER2_ELIGIBLE_PAX  TIER3_ELIGIBLE_PAX
0                    545                7598               28906
1                    876               11897               45356
2                    840               10606               40785
3                   1054               14287               54546
4                   1054               14287               54546
...                  ...                 ...                 ...
9995                1054               14287               54546
9996                 545                7598               28906
9997                 876               11897               45356
9998                 736                9285               35634
9999                 736                9285               35634

[10000 rows x 3 columns]


,TIER_1_%,TIER_2_%,TIER_3_%,total_pax
0,1.5,20.5,78.0,37049
1,1.5,20.5,78.0,58129
2,1.6,20.3,78.1,52231
3,1.5,20.4,78.0,69887
4,1.5,20.4,78.0,69887
...,...,...,...,...
9995,1.5,20.4,78.0,69887
9996,1.5,20.5,78.0,37049
9997,1.5,20.5,78.0,58129
9998,1.6,20.3,78.1,45655


In [22]:
df_tier_proportions = pd.concat([df,proportion_time],axis=1)
dd = df_tier_proportions.groupby(['HAUL','TIME_OF_DAY'])[['TIER_1_%',	'TIER_2_%',	'TIER_3_%']].max()
dd

TIER_1_%  TIER_2_%  TIER_3_%
HAUL  TIME_OF_DAY                              
LONG  Afternoon         1.5      20.5      78.0
      Evening           1.6      20.3      78.1
      Lunchtime         1.3      20.4      78.2
      Morning           1.5      20.5      78.0
SHORT Afternoon         1.6      20.3      78.1
      Evening           1.5      20.4      78.0
      Lunchtime         1.8      20.5      77.8
      Morning           1.6      20.3      78.1

In [23]:
df_tier_proportions.sample(15)

,FLIGHT_DATE,FLIGHT_TIME,TIME_OF_DAY,AIRLINE_CD,FLIGHT_NO,DEPARTURE_STATION_CD,ARRIVAL_STATION_CD,ARRIVAL_COUNTRY,ARRIVAL_REGION,HAUL,...,BUSINESS_CLASS_SEATS,ECONOMY_SEATS,TIER1_ELIGIBLE_PAX,TIER2_ELIGIBLE_PAX,TIER3_ELIGIBLE_PAX,TOTAL_PASSENGERS,TIER_1_%,TIER_2_%,TIER_3_%,total_pax
6801,2025-04-01,07:20:00,Morning,BA,BA5939,LHR,FRA,Germany,Europe,SHORT,...,0,180,0,3,16,180,1.6,20.3,78.1,79850
6769,2025-06-27,15:55:00,Afternoon,BA,BA8187,LHR,ZRH,Switzerland,Europe,SHORT,...,0,180,0,4,20,180,1.6,20.3,78.1,52231
8927,2025-10-06,08:21:00,Morning,BA,BA8716,LHR,MAD,Spain,Europe,SHORT,...,11,169,2,6,26,180,1.6,20.3,78.1,79850
6343,2025-09-05,18:40:00,Evening,BA,BA1909,LHR,JFK,USA,North America,LONG,...,32,300,1,7,28,332,1.6,20.3,78.1,45655
6987,2025-10-05,17:36:00,Afternoon,BA,BA8042,LHR,CDG,France,Europe,SHORT,...,3,177,1,19,63,180,1.6,20.3,78.1,52231
2741,2025-09-11,12:58:00,Lunchtime,BA,BA4092,LHR,AMS,Netherlands,Europe,SHORT,...,12,168,0,10,38,180,1.8,20.5,77.8,30274
5832,2025-08-30,15:49:00,Afternoon,BA,BA6298,LHR,LAX,USA,North America,LONG,...,56,275,1,2,13,331,1.5,20.5,78.0,37049
4199,2025-05-15,20:30:00,Evening,BA,BA1098,LHR,ZRH,Switzerland,Europe,SHORT,...,12,168,0,11,39,180,1.5,20.4,78.0,69887
1703,2025-07-22,12:13:00,Lunchtime,BA,BA7615,LHR,MUC,Germany,Europe,SHORT,...,0,180,1,14,48,180,1.8,20.5,77.8,30274
5751,2025-05-13,20:48:00,Evening,BA,BA7212,LHR,HND,Japan,Asia,LONG,...,97,358,3,1,11,469,1.6,20.3,78.1,45655
